# 04 · Quanto investir em cada ativo: alocação com VQE

**Pergunta:** dado o que aconteceu com os retornos de 2023 e 2024, quanto da carteira deve ir para cada um dos 4 ativos (ITUB4, VALE3, KLBN4, BBAS3)?

Os notebooks anteriores escolhem **quais** ativos entram, com pesos iguais. Aqui queremos **quanto** em cada um. Para isso:

1. Mantemos a mesma lógica matemática da DualQ: função objetivo $q\,w^T\Sigma w - \mu^T w$, restrição de soma e penalidade para virar QUBO.
2. Codificamos o peso de cada ativo com **3 bits**, então temos $4 \times 3 = 12$ qubits.
3. Resolvemos com um **baseline clássico contínuo** (o melhor possível, sem discretizar), com **força bruta** sobre a versão discretizada e com **VQE**.
4. Testamos a carteira escolhida em **2025**, dado que os notebooks anteriores mostraram que a otimização pode ajustar ruído.

Cada seção explica o que está sendo feito antes de fazer. Os circuitos aparecem na seção 7.

## 1. Antes de tudo: o que a referência faz, e o que vamos mudar

A referência (DualQ, "Otimização Revisitada – abordagem restrita binária") escreve:

$$
\min_{x\in\{0,1\}^n} \; q\,x^T\Sigma x - \mu^T x \qquad \text{com } \mathbf{1}^T x = B
$$

Lendo com calma, cada peça significa:

- **$x_i \in \{0,1\}$:** o ativo $i$ entra (1) ou não entra (0) na carteira.
- **$\mu$:** retorno esperado de cada ativo. Aqui, anual.
- **$\Sigma$:** quanto os ativos se movem juntos (covariância). Aqui, anual.
- **$q$:** apetite ao risco. Quanto maior, mais a carteira foge de risco.
- **$\mathbf{1}^T x = B$:** exatamente $B$ ativos entram.

**O que a matemática está certa:** a conversão para QUBO está correta (verificamos no notebook 02) e a penalidade funciona quando $\lambda$ é grande o bastante.

**O que ela não responde:** com $x_i \in \{0,1\}$ e $B$ ativos, a carteira fica com pesos iguais, $1/B$ para cada um. Ela diz *quais* ativos, mas não *quanto*.

**A melhoria:** trocar $x_i$ por um peso $w_i$ em passos de $1/8$:

$$
w_i = \frac{u_i}{8}, \qquad u_i = 1\cdot b_{i,0} + 2\cdot b_{i,1} + 4\cdot b_{i,2}, \qquad b_{i,k}\in\{0,1\}, \qquad \sum_i u_i = 8
$$

Cada ativo recebe 3 bits ($b_{i,0}$, $b_{i,1}$, $b_{i,2}$). Com eles, $u_i$ vai de 0 a 7, ou seja, o peso vai de 0% a 87,5% em passos de 12,5%. A restrição $\sum_i u_i = 8$ garante que a carteira some 100%.

Essa é a mesma ideia de binarização da DualQ, aplicada a pesos em vez de a seleções. A troca tem um custo: a carteira fica discreta, com erro máximo de cerca de 6% por ativo. Medimos esse custo na seção 4.

In [1]:
import warnings
warnings.filterwarnings('ignore', message='.*SparseEfficiency.*')
warnings.filterwarnings('ignore', category=Warning, module='scipy.sparse')

import os
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms.minimum_eigensolvers import SamplingVQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit.circuit.library import n_local, QAOAAnsatz

np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv('../data/precos_b3.csv', header=[0, 1], index_col=0, skiprows=[2], parse_dates=True)
close = df['Close'].apply(pd.to_numeric, errors='coerce').dropna()
close.index = pd.to_datetime(close.index)
ativos = list(close.columns)
n = len(ativos)

def estatisticas(precos):
    """Retorno esperado anual (média diária × 252) e covariância anual (diária × 252)."""
    r = precos.pct_change().dropna()
    return (r.mean() * 252).values, (r.cov() * 252).values

# Treino para escolher a carteira; 2025 fica de fora até a seção 6
mu, Sigma = estatisticas(close.loc['2023-01-01':'2024-12-31'])
mu_t, Sigma_t = estatisticas(close.loc['2025-01-01':'2025-12-31'])

q = 0.5          # apetite ao risco
K = 3            # bits por ativo
SOMA = 8         # 8 unidades de 1/8 = 100% da carteira
N = n * K        # número de qubits

print('Ativos:', ativos)
print('Qubits necessários:', N)
print('\nRetorno esperado anual (treino):', dict(zip(ativos, mu.round(4))))

Ativos: ['BBAS3.SA', 'ITUB4.SA', 'KLBN4.SA', 'VALE3.SA']
Qubits necessários: 12

Retorno esperado anual (treino): {'BBAS3.SA': np.float64(0.2946), 'ITUB4.SA': np.float64(0.1956), 'KLBN4.SA': np.float64(0.2037), 'VALE3.SA': np.float64(-0.1331)}


## 2. O baseline clássico contínuo

Antes de qualquer quântico, precisamos de uma resposta de referência. A melhor carteira possível, sem nenhuma discretização, resolve:

$$
\min_{w} \; q\,w^T\Sigma w - \mu^T w \qquad \text{com } w_i \ge 0, \quad \sum_i w_i = 1
$$

Isso é um problema de otimização quadrática convexa. O `scipy` resolve em milissegundos. Qualquer método quântico precisa ser comparado com essa resposta.

In [2]:
def objetivo_w(w):
    return q * w @ Sigma @ w - mu @ w

res_cont = minimize(
    objetivo_w,
    x0=np.ones(n) / n,
    bounds=[(0, 1)] * n,
    constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}],
    method='SLSQP',
)
w_cont = res_cont.x

print('Convergiu:', res_cont.success)
print('\nPesos ótimos contínuos (baseline):')
print(pd.Series(w_cont, index=ativos).round(4).to_string())
print(f'\nObjetivo contínuo: {objetivo_w(w_cont):.6f}')
print(f'Objetivo com pesos iguais (25% cada): {objetivo_w(np.ones(n)/n):.6f}')

Convergiu: True

Pesos ótimos contínuos (baseline):
BBAS3.SA    1.0
ITUB4.SA    0.0
KLBN4.SA    0.0
VALE3.SA    0.0

Objetivo contínuo: -0.272554
Objetivo com pesos iguais (25% cada): -0.129729


## 3. Montando o QUBO com pesos discretos

Queremos escrever tudo em função dos bits $b$ (um vetor de 12 posições). Para isso, criamos uma matriz $M$ de tamanho $4\times 12$ que transforma bits em pesos:

$$
w = M\,b, \qquad M_{i,(i,k)} = \frac{2^k}{8}
$$

Com isso, a função objetivo fica:

$$
f(b) = q\,b^T M^T \Sigma M b - \mu^T M b + \lambda\,(\mathbf{1}^T M b - 1)^2
$$

Expandindo a penalidade (como no notebook 02, usando $b_j^2 = b_j$):

$$
Q = q\,M^T\Sigma M + \lambda\,a\,a^T, \qquad c = -M^T\mu - 2\lambda\,a, \qquad d = \lambda, \qquad \text{com } a = M^T\mathbf{1}
$$

Conferimos numericamente que essa forma matricial é igual à forma com pesos, para todos os vetores de bits.

In [3]:
M = np.zeros((n, N))
for i in range(n):
    for k in range(K):
        M[i, i * K + k] = 2 ** k / 8

def pesos_de_bits(b):
    return M @ b

def unidades(b):
    return (np.asarray(b).reshape(n, K) * (2 ** np.arange(K))).sum(axis=1).astype(int)

def build_qubo(lam):
    um = np.ones(n)
    a = M.T @ um
    Q = q * M.T @ Sigma @ M + lam * np.outer(a, a)
    c = -M.T @ mu - 2 * lam * a
    d = lam
    return Q, c, d

def build_qp(lam):
    Q, c, d = build_qubo(lam)
    qp = QuadraticProgram(f'alocacao_lam{lam}')
    for j in range(N):
        qp.binary_var(name=f'b{j}')
    qp.minimize(constant=d, linear=c, quadratic=Q)
    return qp

estados = np.array(list(product([0, 1], repeat=N)))   # 4096 vetores de bits
viavel = np.array([unidades(b).sum() == SOMA for b in estados])

# Verificação: forma matricial == forma com pesos
lam = 10.0
Q, c, d = build_qubo(lam)
def f_pesos(b, lam):
    w = pesos_de_bits(b)
    return q * w @ Sigma @ w - mu @ w + lam * (w.sum() - 1) ** 2

erro = max(abs(b @ Q @ b + c @ b + d - f_pesos(b, lam)) for b in estados[::37])
print(f'Erro entre forma matricial e forma com pesos: {erro:.2e}')
assert erro < 1e-12

print(f'\nTotal de vetores de bits: {len(estados)}')
print(f'Alocações viáveis (soma dos pesos = 100%): {viavel.sum()}')
print(f'Chute uniforme acerta viável com probabilidade: {viavel.mean():.4f}')

Erro entre forma matricial e forma com pesos: 1.78e-14

Total de vetores de bits: 4096
Alocações viáveis (soma dos pesos = 100%): 161
Chute uniforme acerta viável com probabilidade: 0.0393


## 4. Força bruta: a melhor alocação discreta

Com 4096 vetores de bits, é possível testar todos. Isso nos dá a melhor alocação discreta exata, a referência correta para o VQE. Também comparamos com o baseline contínuo, para medir o custo da discretização.

Antes, escolhemos $\lambda$ com um teste simples: se $\lambda$ for pequeno demais, a menor energia pode ser de uma carteira que viola a soma dos pesos. O teste abaixo mostra que $\lambda = 1$ **não basta**: o mínimo global ainda é inviável. Escolhemos $\lambda = 10$, o menor valor testado que resolve.

Vale notar um fato útil: a penalidade é zero para toda carteira viável, então a **melhor carteira viável não depende de $\lambda$**. Um $\lambda$ grande só garante que o solver quântico, que olha a menor energia, não devolva uma carteira inviável.

In [4]:
def energias_qubo(lam):
    Q, c, d = build_qubo(lam)
    return np.einsum('ij,jk,ik->i', estados, Q, estados) + estados @ c + d

print('Teste de λ: o mínimo global é viável?\n')
for lam_t in [0.01, 0.1, 1.0, 10.0]:
    e = energias_qubo(lam_t)
    k = np.argmin(e)
    print(f'λ = {lam_t:<6} mínimo viável: {viavel[k]!s:<5}  pesos = {pesos_de_bits(estados[k])}')

lam = 10.0
e = energias_qubo(lam)
idx_viaveis = np.where(viavel)[0]
k_ot = idx_viaveis[np.argmin(e[idx_viaveis])]
b_ot = estados[k_ot]
optimos = [estados[i] for i in idx_viaveis if abs(e[i] - e[k_ot]) < 1e-9]

print(f'\nMelhor alocação discreta viável (λ = {lam}):')
print('  unidades u:', unidades(b_ot), '| soma =', unidades(b_ot).sum())
print('  pesos w   :', pesos_de_bits(b_ot))
print('  ótimos empatados:', len(optimos))
print('  objetivo (sem penalidade):', round(objetivo_w(pesos_de_bits(b_ot)), 6))
print(f'\nBaseline contínuo: {objetivo_w(w_cont):.6f}')
print(f'Custo da discretização: {objetivo_w(pesos_de_bits(b_ot)) - objetivo_w(w_cont):.6f}')

Teste de λ: o mínimo global é viável?

λ = 0.01   mínimo viável: False  pesos = [0.875 0.875 0.875 0.   ]
λ = 0.1    mínimo viável: False  pesos = [0.875 0.125 0.75  0.   ]
λ = 1.0    mínimo viável: False  pesos = [0.875 0.    0.25  0.   ]
λ = 10.0   mínimo viável: True   pesos = [0.875 0.    0.125 0.   ]

Melhor alocação discreta viável (λ = 10.0):
  unidades u: [7 0 1 0] | soma = 8
  pesos w   : [0.875 0.    0.125 0.   ]
  ótimos empatados: 1
  objetivo (sem penalidade): -0.265214

Baseline contínuo: -0.272554
Custo da discretização: 0.007339


In [5]:
comparacao = pd.DataFrame({
    'contínuo (baseline)': w_cont,
    'discreto (força bruta)': pesos_de_bits(b_ot),
    'pesos iguais': np.ones(n) / n,
}, index=ativos)
comparacao.round(4)

,contínuo (baseline),discreto (força bruta),pesos iguais
BBAS3.SA,1.0,0.875,0.25
ITUB4.SA,0.0,0.000,0.25
KLBN4.SA,0.0,0.125,0.25
VALE3.SA,0.0,0.000,0.25


A tabela mostra a diferença entre a melhor carteira possível e a melhor carteira com passos de 12,5%. Se a diferença for pequena, a discretização não atrapalha a decisão. Se for grande, precisamos de mais bits.

## 5. VQE: o algoritmo quântico variacional

**Ideia central, em linguagem simples:**

1. Montamos um circuito com parâmetros $\theta$ (ângulos de rotação). Esse circuito é o **ansatz**.
2. Medimos o circuito e obtemos uma distribuição de bitstrings.
3. Calculamos a energia média dessas medições, usando a função objetivo do QUBO.
4. Um otimizador clássico (aqui, COBYLA) ajusta $\theta$ para diminuir essa energia.
5. Repetimos até convergir. A bitstring mais provável (ou de menor energia) é a resposta.

O ansatz usado aqui é o mesmo da DualQ: rotações RY em cada qubit, seguidas de emaranhamento CZ entre todos os pares (`entanglement='full'`). A DualQ usa 11 repetições desse bloco. Como nosso problema tem 12 qubits, cada repetição custa mais, então testamos 1, 2 e 3 repetições e comparamos.

**Por que VQE e não só QAOA?** O QAOA tem um circuito com estrutura fixa, derivada do Hamiltoniano. O VQE usa um circuito genérico, que pode ser mais expressivo, mas também tem mais parâmetros para o otimizador ajustar. Os dois são variacionais; a diferença está na forma do circuito.

In [6]:
def rodar_vqe(reps, seed, lam_t=10.0, maxiter=1000):
    qp_t = build_qp(lam_t)
    np.random.seed(seed)
    sampler = StatevectorSampler(default_shots=4096, seed=seed)
    ansatz = n_local(N, ['ry'], ['cz'], entanglement='full', reps=reps)
    # Ponto inicial semeado: sem isso, o VQE sorteia os ângulos com outro gerador e o resultado muda a cada execução
    ponto_inicial = np.random.default_rng(seed).uniform(-np.pi, np.pi, ansatz.num_parameters)
    vqe = SamplingVQE(sampler=sampler, ansatz=ansatz, optimizer=COBYLA(maxiter=maxiter),
                      initial_point=ponto_inicial)
    res = MinimumEigenOptimizer(vqe).solve(qp_t)

    p_opt = sum(s.probability for s in res.samples if any(np.array_equal(s.x, o) for o in optimos))
    p_viav = sum(s.probability for s in res.samples if unidades(s.x).sum() == SOMA)
    return {
        'reps': reps,
        'seed': seed,
        'parâmetros': ansatz.num_parameters,
        'avaliações': res.min_eigen_solver_result.cost_function_evals,
        'acertou solver': any(np.array_equal(res.x, o) for o in optimos),
        'P(ótimo)': p_opt,
        'P(viável)': p_viav,
        'res': res,
    }

In [7]:
linhas = []
resultados_vqe = {}
for reps in [1, 2, 3]:
    for seed in [0, 1, 2]:
        r = rodar_vqe(reps, seed)
        resultados_vqe[(reps, seed)] = r
        linhas.append({k: r[k] for k in ['reps', 'seed', 'parâmetros', 'avaliações',
                                          'acertou solver', 'P(ótimo)', 'P(viável)']})

tabela_vqe = pd.DataFrame(linhas)
print(tabela_vqe.round(5).to_string(index=False))
av = tabela_vqe['avaliações']
print(f'\nAvaliações do COBYLA: de {av.min()} a {av.max()} (limite = 1000).')
print(f'Execuções que acertaram a melhor alocação viável: {int(tabela_vqe["acertou solver"].sum())} de {len(tabela_vqe)}.')
print(f'\nChute uniforme: P(viável) = {viavel.mean():.4f} | P(ótimo) = {1/len(estados) * len(optimos):.6f}')

 reps  seed  parâmetros  avaliações  acertou solver  P(ótimo)  P(viável)
    1     0          24         308           False   0.00000    0.53408
    1     1          24         322           False   0.00000    0.00448
    1     2          24         379            True   0.00000    0.21119
    2     0          36         279           False   0.00000    0.00081
    2     1          36         436           False   0.00000    0.21291
    2     2          36         363            True   0.00003    0.21066
    3     0          48         401           False   0.00000    0.00102
    3     1          48         540           False   0.00000    0.00947
    3     2          48         421           False   0.00000    0.00298

Avaliações do COBYLA: de 279 a 540 (limite = 1000).
Execuções que acertaram a melhor alocação viável: 2 de 9.

Chute uniforme: P(viável) = 0.0393 | P(ótimo) = 0.000244


**Leitura do resultado (o mais importante desta seção):**

- Em várias execuções, o VQE **aumenta** a probabilidade de carteira viável em relação ao sorteio (até cerca de 53%, contra 3,9% do sorteio). Ou seja, ele aprende a respeitar a soma dos pesos.
- O solver devolveu a melhor alocação viável em 2 de 9 execuções. Mas, nesses casos, a probabilidade do ótimo no circuito é de cerca de 0,003%, abaixo do sorteio (cerca de 0,024%). O acerto veio de uma amostra de menor energia, não de uma distribuição concentrada no ótimo. É o mesmo padrão que vimos no QAOA.
- O COBYLA parou entre 279 e 540 avaliações, antes do limite de 1000. Isso indica convergência por tolerância, não necessariamente um mínimo global. Mais iterações, outras inicializações ou outro otimizador são as próximas tentativas, sem garantia de sucesso.

**Como ler a tabela:**

- `acertou solver`: o VQE devolveu a melhor alocação discreta?
- `P(ótimo)` e `P(viável)`: quanta probabilidade o circuito colocou nas respostas certas. Essas são as métricas que importam, como vimos no notebook 03.
- Compare com o chute uniforme: sortear um vetor de bits acerta uma alocação viável com probabilidade de cerca de 4%. Se o VQE não superar isso de forma clara, o circuito não está aprendendo.

## 6. Qual carteira escolher? E ela funciona fora da amostra?

Escolhemos a carteira com a melhor energia entre as alocações viáveis do VQE (com o mesmo critério do solver), e comparamos com as outras opções. Em seguida, testamos todas em 2025, que não foi usado na escolha.

Usamos o Sharpe com $r_f = 0{,}13$, a aproximação do CDI do README, que ainda precisa ser conferida no Banco Central.

In [8]:
#  Melhor amostra VIÁVEL entre todas as execuções do VQE (menor energia do QUBO)
candidatos = [(s, r) for r in resultados_vqe.values() for s in r['res'].samples
              if unidades(s.x).sum() == SOMA]
if candidatos:
    s_best, r_best = min(candidatos, key=lambda t: t[0].fval)
    w_vqe = pesos_de_bits(np.asarray(s_best.x))
    print(f"Melhor amostra viável do VQE: reps={r_best['reps']}, seed={r_best['seed']}")
else:
    w_vqe = None
    print('Nenhuma execução do VQE produziu uma amostra viável.')
w_bf = pesos_de_bits(b_ot)
w_igual = np.ones(n) / n
rf = 0.13

def resumo_fora_amostra(w):
    ret = w @ mu_t
    vol = np.sqrt(w @ Sigma_t @ w)
    return {'retorno 2025': ret, 'volatilidade 2025': vol, 'Sharpe 2025': (ret - rf) / vol}

opcoes = {
    'pesos iguais': w_igual,
    'contínuo (baseline)': w_cont,
    'discreto (força bruta)': w_bf,
}
if w_vqe is not None:
    opcoes['VQE (melhor amostra viável)'] = w_vqe
fora = pd.DataFrame({nome: resumo_fora_amostra(w) for nome, w in opcoes.items()}).T
pesos_df = pd.DataFrame(opcoes, index=ativos).T

print('Pesos escolhidos com treino (2023–2024):')
print(pesos_df.round(4).to_string())
print('\nDesempenho fora da amostra (2025):')
print(fora.round(4).to_string())

Melhor amostra viável do VQE: reps=1, seed=2
Pesos escolhidos com treino (2023–2024):
                             BBAS3.SA  ITUB4.SA  KLBN4.SA  VALE3.SA
pesos iguais                    0.250      0.25     0.250      0.25
contínuo (baseline)             1.000      0.00     0.000      0.00
discreto (força bruta)          0.875      0.00     0.125      0.00
VQE (melhor amostra viável)     0.875      0.00     0.125      0.00

Desempenho fora da amostra (2025):
                             retorno 2025  volatilidade 2025  Sharpe 2025
pesos iguais                       0.2246             0.1517       0.6237
contínuo (baseline)               -0.0008             0.2798      -0.4673
discreto (força bruta)            -0.0088             0.2512      -0.5525
VQE (melhor amostra viável)       -0.0088             0.2512      -0.5525


**Leitura honesta:** a carteira é escolhida com dados de 2023–2024 e avaliada em 2025. Se a carteira otimizada não superar os pesos iguais em 2025, isso indica que a estimativa de $\mu$ com dois anos de dados não basta, e o problema é a estimativa, não o método de otimização. Isso é o mesmo resultado que apareceu no notebook 01. Não tire conclusões de um único ano.

## 7. Os circuitos

Mostramos os circuitos com 4 qubits, porque com 12 qubits o desenho fica ilegível. A estrutura é a mesma.

**Ansatz do VQE (uma repetição):**

- Cada qubit recebe uma **rotação RY** com um ângulo $\theta$ que o otimizador ajusta.
- Depois, uma camada **CZ** conecta todos os pares de qubits, criando emaranhamento.
- Uma última camada de rotações RY fecha o bloco.

Com 3 repetições, o bloco se repete 3 vezes. O número de parâmetros é 4 × (repetições + 1).

In [9]:
ansatz_4q = n_local(4, ['ry'], ['cz'], entanglement='full', reps=1)
print('Parâmetros:', ansatz_4q.num_parameters)
fig = ansatz_4q.draw('mpl', style='iqp', fold=-1)
os.makedirs('figuras', exist_ok=True)
fig.savefig('figuras/vqe_ansatz_4q_reps1.png', dpi=150, bbox_inches='tight')
plt.show()

Parâmetros: 8


**Circuito QAOA para o mesmo tipo de problema (4 qubits, uma camada):**

O QAOA não tem um ansatz genérico. Ele constrói o circuito a partir do Hamiltoniano de Ising $H$, em duas partes:

- **Camada de custo** $e^{-i\gamma H}$: portas RZZ e RZ, uma para cada termo de $H$. Ela faz a energia da solução aparecer na fase.
- **Camada de mistura** $e^{-i\beta \sum_i X_i}$: portas RX em cada qubit. Ela espalha a probabilidade entre as soluções.

Para mostrar isso, usamos o QUBO de 4 ativos (a formulação do notebook 02), com $B = 2$.

In [10]:
from qiskit_optimization import QuadraticProgram as QP

ones4 = np.ones(n)
Q2 = q * Sigma + 1.0 * np.outer(ones4, ones4)
c2 = -mu - 2 * 1.0 * 2 * ones4
qp2 = QP('selecao_4ativos')
for a in ativos:
    qp2.binary_var(name=a)
qp2.minimize(constant=1.0 * 4, linear=c2, quadratic=Q2)

H4, off4 = qp2.to_ising()
qaoa_4q = QAOAAnsatz(cost_operator=H4, reps=1)
print('Parâmetros do QAOA (γ e β):', qaoa_4q.num_parameters)
fig2 = qaoa_4q.decompose(reps=1).draw('mpl', style='iqp', fold=-1)
fig2.savefig('figuras/qaoa_4q_reps1.png', dpi=150, bbox_inches='tight')
plt.show()

Parâmetros do QAOA (γ e β): 2


## 8. O que este notebook responde, e o que não responde

**Resposta à pergunta da seção 0, com os dados atuais:**

- Com o treino de 2023–2024, o baseline contínuo concentra tudo em BBAS3. Essa é a resposta do método, e ela é instável: é um canto da região viável, impulsionado por um retorno esperado alto em dois anos.
- A força bruta discreta chega a 87,5% em BBAS3 e 12,5% em KLBN4, com custo de discretização pequeno (seção 4).
- O VQE devolveu essa alocação em 2 de 9 execuções, mas com probabilidade de ~0,003% no circuito, abaixo do sorteio. Em várias execuções ele respeitou a restrição de soma (até ~53% de viabilidade), mas sem concentrar o resultado no ótimo.
- Em 2025, pesos iguais tiveram o melhor Sharpe. As alternativas concentradas ficaram abaixo, incluindo a escolha do VQE.
- **Conclusão honesta:** com dois anos de treino e um de teste, este método não mostra evidência de que otimizar supere pesos iguais. A carteira diversificada foi a escolha mais robusta nos dados disponíveis.

**Responde:**

1. Qual a melhor alocação contínua para o período de treino (baseline).
2. Qual a melhor alocação com passos de 12,5%, por força bruta, e quanto a discretização custa.
3. Se o VQE consegue encontrar essa alocação, e com que probabilidade, comparado ao sorteio.
4. Como as alternativas se comportam em 2025, fora da amostra.

**Não responde:**

- Se a carteira escolhida vai render no futuro. Um ano de teste não basta.
- Se o VQE tem vantagem sobre o clássico. Com 12 qubits, a força bruta resolve em frações de segundo.
- O efeito de ruído de hardware real. As simulações são ideais.

**Limitações da formulação que você deve conhecer:**

- A penalidade $\lambda$ domina a parte de risco se for grande, e não basta se for pequena. A seção 4 mostra o teste.
- $\mu$ estimado com dois anos é ruidoso. O QUBO herda esse ruído.
- A discretização em 12,5% é grosseira para carteiras com peso pequeno. Mais bits resolvem, mas aumentam o número de qubits.